# WP4: Invariant manifolds and application to Mars exploration mission

## Personal problem specification 

| Parameter | Value |
|-----------|-------|
| Sun/Earth Lagrange point | L2 |
| Sun/Mars Lagrange point | L2 |
| Shift in Lagrange points | 0.00100 |
| Radius Martian parking orbit | 244,444 km |
| Lightness number | 0.0300 |

## Library & constant imports

In [1]:
from constants import *

import numpy as np
import scipy 
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch
import pickle
import nbimporter
from IPython.display import display

from wp1 import dUdx_colinear, nablaU
from auxiliary_functions import nablaU, dUdx_colinear, hessianU, jacobian, eom

# ── Font sizes for plots (adjustable) ──────────────────────────────────────────
FONTSIZE_SMALL = 16              # Contour labels, small text
FONTSIZE_TICKS = 15              # Tick labels
FONTSIZE_AXIS_LABELS = 30        # Axis point labels
FONTSIZE_XLABEL_YLABEL = 16      # Axis label text
FONTSIZE_LEGEND = 15             # Legend and annotations
FONTSIZE_TITLE = 36              # Main titles

Importing of the eigenvalues and eigenvectors saved from the notebook for WP3. 

In [2]:
with open('eigenvalues_eigenvectors.pkl', 'rb') as f:
    eigenvalues_eigenvectors = pickle.load(f)

eigenvalues_mars_classical = eigenvalues_eigenvectors['eigenvalues_mars_classical']
eigenvectors_mars_classical = eigenvalues_eigenvectors['eigenvectors_mars_classical']

And also importing the Lagrange points position.

In [3]:
with open('lagrange_points.pkl', 'rb') as f:
    lagrange_points = pickle.load(f)

# L2_earth = lagrange_points['L2_earth']
L2_mars  = lagrange_points['L2_mars']

## Task 4.1

Computing and plotting the unstable and stable manifolds for the **classical Sun-Mars Lagrange point**, using the following parameters provided in the assignment text. 

In [4]:
earth_years = 5 
tolerance = 1e-12 # absolute and relative integration tolerance
perturbation_magnitude = 1e-5 

time_span = earth_years * year_in_seconds / time_unit_sun_mars

It implies the integration of the stable and unstable equations of motion. First, the unstable one will be considered. 

### Unstable manifold

In [5]:
unstable_eigenvalue_mars = eigenvalues_mars_classical[0].real
unstable_eigenvector_mars = eigenvectors_mars_classical[:, 0].real 
print("Unstable eigenvalue:", unstable_eigenvalue_mars)
print("Unstable eigenvector:", unstable_eigenvector_mars)

Unstable eigenvalue: 2.4969462455360953
Unstable eigenvector: [-0.32680059  0.17726628 -0.8160035   0.44262437]


Once checked that the values are the one corresponding to the unstable manifold (eigenvalue with positive real part), the EOM are integrated.

In [6]:
state0_unstable = np.array([L2_mars, 0.0, 0.0, 0.0]) + perturbation_magnitude * unstable_eigenvector_mars

sol_unstable = solve_ivp(eom, [0, time_span], state0_unstable, args=(mass_ratio_sun_mars,), method='DOP853', rtol=tolerance, atol=tolerance, dense_output=True)

### Stable manifold

In [7]:
stable_eigenvalue_mars = eigenvalues_mars_classical[1].real
stable_eigenvector_mars = eigenvectors_mars_classical[:, 1].real 
print("Stable eigenvalue:", stable_eigenvalue_mars)
print("Stable eigenvector:", stable_eigenvector_mars)

Stable eigenvalue: -2.496946245536095
Stable eigenvector: [-0.32680059 -0.17726628  0.8160035   0.44262437]


Once checked that the values are the one corresponding to the stable manifold (eigenvalue with negative real part), the EOM are integrated. Mind that the integration interval now goes from $[-t, 0]$ (the opposite as before). 

In [8]:
state0_stable = np.array([L2_mars, 0.0, 0.0, 0.0]) + perturbation_magnitude * stable_eigenvector_mars

sol_stable = solve_ivp(eom, [-time_span, 0], state0_stable, args=(mass_ratio_sun_mars,), method='DOP853', rtol=tolerance, atol=tolerance, dense_output=True)